In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

import optuna
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_validate
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

from config.paths import config

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

In [2]:
processed_data = joblib.load(config.processed_data_dir / "feature_engineered_data.pkl")
X_train = processed_data['X_train']
X_test = processed_data['X_test']
y_train = processed_data['y_train']
y_test = processed_data['y_test']

feature_names = processed_data['feature_names']
with open(config.reports_dir / 'fast_baseline_models_results.json', 'r') as f:
    baseline_results = json.load(f)
print(f"Training data: {X_train.shape}")
print(f"Testing data: {X_test.shape}")

Training data: (156156, 82)
Testing data: (39040, 82)


In [3]:
performance_data = []
for model_data in baseline_results['performance_summary']:
    accuracy_str = model_data['Accuracy'].split(' ± ')[0]
    roc_auc_str = model_data['ROC-AUC'].split(' ± ')[0]
    training_time_str = model_data['Training Time (s)']
    performance_data.append({
        'model': model_data['Model'],
        'accuracy': float(accuracy_str),
        'roc_auc': float(roc_auc_str),
        'training_time': float(training_time_str)
    })
performance_df = pd.DataFrame(performance_data)
performance_df['overall_score'] = (performance_df['accuracy'] * 0.4 +
                                  performance_df['roc_auc'] * 0.4 +
                                  (1 / performance_df['training_time']) * 0.2)

all_models = performance_df['model'].tolist()

print("ALL MODELS AVAILABLE FOR TUNING:")
for i, (idx, row) in enumerate(performance_df.iterrows()):
    print(f"{i+1}. {row['model']}: Accuracy: {row['accuracy']:.4f}, ROC-AUC: {row['roc_auc']:.4f}")

ALL MODELS AVAILABLE FOR TUNING:
1. Logistic Regression: Accuracy: 0.9741, ROC-AUC: 0.9960
2. Random Forest: Accuracy: 0.9759, ROC-AUC: 0.9962
3. XGBoost: Accuracy: 0.9832, ROC-AUC: 0.9986
4. LightGBM: Accuracy: 0.9832, ROC-AUC: 0.9986
5. Linear SVM: Accuracy: 0.9729, ROC-AUC: 0.9961
6. KNN: Accuracy: 0.9339, ROC-AUC: 0.9652


In [4]:
xgb_param_space = {
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 6, 9],
    'n_estimators': [100, 200, 300],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [0, 0.1, 1],
    'min_child_weight': [1, 3, 5]
}

rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

lgb_param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [31, 63, 127, 255],
    'max_depth': [-1, 10, 20, 30],
    'min_child_samples': [20, 50, 100],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.1, 0.5, 1],
    'reg_lambda': [0, 0.1, 0.5, 1]
}

lr_param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2', 'elasticnet'],
    'solver': ['liblinear', 'saga'],
    'max_iter': [1000, 2000]
}

svm_param_dist = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto', 0.1, 0.01],
    'degree': [2, 3, 4],
    'probability': [True]
}

knn_param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

In [5]:
def objective_xgb(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 1),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 5),
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'logloss',
        'use_label_encoder': False
    }

    model = XGBClassifier(**params)
    cv_scores = cross_validate(model, X_train, y_train, cv=2, scoring='roc_auc', n_jobs=-1)
    return np.mean(cv_scores['test_score'])

In [6]:
optimized_models = {}
optimization_results = {}
tuning_results = {}

cv_fast = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print("=" * 50)
print("HYPERPARAMETER OPTIMIZATION STARTED")
print("=" * 50)

print("SKIPPING SVM AND KNN - TOO SLOW FOR LARGE DATASET")
models_to_use = [model for model in all_models if model not in ['Linear SVM', 'KNN']]
print(f"Optimizing models: {models_to_use}")
print()

optuna.logging.set_verbosity(optuna.logging.WARNING)

if 'XGBoost' in models_to_use:
    print("XGBoost - Bayesian Optimization (10 trials)")
    print("-" * 40)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective_xgb, n_trials=10, show_progress_bar=False)

    best_xgb_params = study.best_params
    optimized_models['XGBoost'] = XGBClassifier(**best_xgb_params)
    xgb_scores = cross_validate(optimized_models['XGBoost'], X_train, y_train, cv=cv_fast, scoring='roc_auc')
    optimization_results['XGBoost'] = np.mean(xgb_scores['test_score'])

    print(f"Best ROC-AUC: {study.best_value:.6f}")
    print(f"Final Score: {optimization_results['XGBoost']:.4f}")
    print()

if 'Random Forest' in models_to_use:
    print("Random Forest - Fast Tuning")
    print("-" * 40)

    rf_fast_params = {
        'n_estimators': 200,
        'max_depth': 20,
        'min_samples_split': 5,
        'min_samples_leaf': 2,
        'max_features': 'sqrt',
        'random_state': 42,
        'n_jobs': -1
    }
    optimized_models['Random Forest'] = RandomForestClassifier(**rf_fast_params)
    rf_scores = cross_validate(optimized_models['Random Forest'], X_train, y_train, cv=cv_fast, scoring='roc_auc')
    optimization_results['Random Forest'] = np.mean(rf_scores['test_score'])

    print(f"Final Score: {optimization_results['Random Forest']:.4f}")
    print()

if 'LightGBM' in models_to_use:
    print("LightGBM - Fast Tuning")
    print("-" * 40)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning)

        lgb_fast_params = {
            'n_estimators': 200,
            'learning_rate': 0.1,
            'num_leaves': 63,
            'max_depth': -1,
            'random_state': 42,
            'n_jobs': -1,
            'verbosity': -1
        }
        optimized_models['LightGBM'] = LGBMClassifier(**lgb_fast_params)
        lgb_scores = cross_validate(optimized_models['LightGBM'], X_train, y_train, cv=cv_fast, scoring='roc_auc')
        optimization_results['LightGBM'] = np.mean(lgb_scores['test_score'])

    print(f"Final Score: {optimization_results['LightGBM']:.4f}")
    print()

if 'Logistic Regression' in models_to_use:
    print("Logistic Regression - Fast Tuning")
    print("-" * 40)

    lr_fast_params = {
        'C': 1.0,
        'penalty': 'l2',
        'solver': 'liblinear',
        'random_state': 42,
        'n_jobs': -1,
        'max_iter': 1000
    }
    optimized_models['Logistic Regression'] = LogisticRegression(**lr_fast_params)
    lr_scores = cross_validate(optimized_models['Logistic Regression'], X_train, y_train, cv=cv_fast, scoring='roc_auc')
    optimization_results['Logistic Regression'] = np.mean(lr_scores['test_score'])

    print(f"Final Score: {optimization_results['Logistic Regression']:.4f}")
    print()

HYPERPARAMETER OPTIMIZATION STARTED
SKIPPING SVM AND KNN - TOO SLOW FOR LARGE DATASET
Optimizing models: ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM']

XGBoost - Bayesian Optimization (10 trials)
----------------------------------------
Best ROC-AUC: 0.998631
Final Score: 0.9986

Random Forest - Fast Tuning
----------------------------------------
Final Score: 0.9963

LightGBM - Fast Tuning
----------------------------------------
Final Score: 0.9985

Logistic Regression - Fast Tuning
----------------------------------------
Final Score: 0.9959



In [7]:
print("=" * 50)
print("MODEL EVALUATION")
print("=" * 50)

for model_name, model in optimized_models.items():
    print(f"Training {model_name}...")
    start_time = time.time()

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")

        cv_scores = cross_validate(
            model, X_train, y_train,
            cv=cv_strategy,
            scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
            n_jobs=-1,
            return_train_score=False
        )

        model.fit(X_train, y_train)
        training_time = time.time() - start_time

        tuning_results[model_name] = {
            'model': model,
            'training_time': training_time,
            'cv_scores': cv_scores,
            'params': model.get_params()
        }

    roc_auc = np.mean(cv_scores['test_roc_auc'])
    print(f"ROC-AUC: {roc_auc:.4f} | Time: {training_time:.1f}s")
    print()

MODEL EVALUATION
Training XGBoost...
ROC-AUC: 0.9987 | Time: 11.1s

Training Random Forest...
ROC-AUC: 0.9966 | Time: 24.9s

Training LightGBM...
ROC-AUC: 0.9986 | Time: 12.0s

Training Logistic Regression...
ROC-AUC: 0.9959 | Time: 21.9s



In [9]:
print("=" * 50)
print("ENSEMBLE METHODS")
print("=" * 50)

class WeightedAverageEnsemble:
    def __init__(self, models, weights):
        self.models = models
        self.weights = weights

    def fit(self, X, y):
        for model in self.models:
            model.fit(X, y)
        return self

    def predict_proba(self, X):
        probas = [model.predict_proba(X) for model in self.models]
        weighted_proba = np.average(probas, axis=0, weights=self.weights)
        return weighted_proba

    def predict(self, X):
        proba = self.predict_proba(X)
        return (proba[:, 1] > 0.5).astype(int)

print("CREATING WEIGHTED AVERAGE ENSEMBLE...")
weights = [optimization_results.get(name, 0.5) for name in optimized_models.keys()]
weights = [w/sum(weights) for w in weights]
weighted_ensemble = WeightedAverageEnsemble(
    models=list(optimized_models.values()),
    weights=weights
)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    weighted_ensemble.fit(X_train, y_train)

weighted_scores = {'test_accuracy': [], 'test_precision': [], 'test_recall': [], 'test_f1': [], 'test_roc_auc': []}
for train_idx, val_idx in cv_strategy.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        weighted_ensemble.fit(X_tr, y_tr)
        y_pred_proba = weighted_ensemble.predict_proba(X_val)[:, 1]
        y_pred = (y_pred_proba > 0.5).astype(int)

    weighted_scores['test_accuracy'].append(accuracy_score(y_val, y_pred))
    weighted_scores['test_precision'].append(precision_score(y_val, y_pred))
    weighted_scores['test_recall'].append(recall_score(y_val, y_pred))
    weighted_scores['test_f1'].append(f1_score(y_val, y_pred))
    weighted_scores['test_roc_auc'].append(roc_auc_score(y_val, y_pred_proba))

tuning_results['Weighted_Ensemble'] = {
    'model': weighted_ensemble,
    'training_time': 0,
    'cv_scores': weighted_scores,
    'params': 'Weighted Average'
}

print(f"Weighted Ensemble ROC-AUC: {np.mean(weighted_scores['test_roc_auc']):.4f}")
print()

ENSEMBLE METHODS
CREATING WEIGHTED AVERAGE ENSEMBLE...
Weighted Ensemble ROC-AUC: 0.9982



In [10]:
print("=" * 50)
print("FINAL MODEL SELECTION")
print("=" * 50)

best_model_name = None
best_roc_auc = 0
best_model_details = {}

for model_name, result in tuning_results.items():
    cv_scores = result['cv_scores']
    roc_auc = np.mean(cv_scores['test_roc_auc'])

    if roc_auc > best_roc_auc:
        best_roc_auc = roc_auc
        best_model_name = model_name
        best_model_details = {
            'roc_auc': roc_auc,
            'accuracy': np.mean(cv_scores['test_accuracy']),
            'f1': np.mean(cv_scores['test_f1']),
            'training_time': result['training_time']
        }

best_model = tuning_results[best_model_name]['model']

print(f"BEST MODEL: {best_model_name}")
print("-" * 30)
print(f"Validation ROC-AUC: {best_model_details['roc_auc']:.6f}")
print(f"Validation Accuracy: {best_model_details['accuracy']:.6f}")
print(f"Validation F1-Score: {best_model_details['f1']:.6f}")
print(f"Training Time: {best_model_details['training_time']:.1f}s")
print()

print("TEST SET EVALUATION:")
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

test_metrics = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

print(f"Test ROC-AUC:  {test_metrics['roc_auc']:.6f}")
print(f"Test Accuracy: {test_metrics['accuracy']:.6f}")
print(f"Test F1-Score: {test_metrics['f1']:.6f}")
print()

print("PERFORMANCE COMPARISON:")
print(f"{'Metric':<12} {'Validation':<12} {'Test':<12} {'Diff':<10}")
print("-" * 50)
for metric in ['roc_auc', 'accuracy', 'f1']:
    val_score = best_model_details[metric]
    test_score = test_metrics[metric]
    diff = test_score - val_score
    diff_color = "✓" if abs(diff) < 0.01 else "⚠" if abs(diff) < 0.05 else "✗"
    print(f"{metric.upper():<12} {val_score:<12.6f} {test_score:<12.6f} {diff:+.6f} {diff_color}")

FINAL MODEL SELECTION
BEST MODEL: XGBoost
------------------------------
Validation ROC-AUC: 0.998692
Validation Accuracy: 0.985886
Validation F1-Score: 0.976060
Training Time: 11.1s

TEST SET EVALUATION:
Test ROC-AUC:  0.998875
Test Accuracy: 0.986885
Test F1-Score: 0.977735

PERFORMANCE COMPARISON:
Metric       Validation   Test         Diff      
--------------------------------------------------
ROC_AUC      0.998692     0.998875     +0.000183 ✓
ACCURACY     0.985886     0.986885     +0.000999 ✓
F1           0.976060     0.977735     +0.001675 ✓


In [12]:
print("=" * 50)
print("SAVING RESULTS")
print("=" * 50)

tuning_output = {
    'best_model': {
        'name': best_model_name,
        'model': best_model,
        'validation_performance': best_model_details,
        'test_performance': test_metrics
    },
    'all_models': {
        model_name: {
            'model': result['model'],
            'performance': {
                'roc_auc': np.mean(result['cv_scores']['test_roc_auc']),
                'accuracy': np.mean(result['cv_scores']['test_accuracy']),
                'f1': np.mean(result['cv_scores']['test_f1'])
            }
        }
        for model_name, result in tuning_results.items()
    },
    'comparison_results': comparison_results
}

joblib.dump(tuning_output, config.models_dir / 'final_tuned_models.pkl')
comparison_df.to_csv(config.reports_dir / 'model_comparison.csv', index=False)

print("RESULTS SAVED:")
print(f"- Models: {config.models_dir / 'final_tuned_models.pkl'}")
print(f"- Comparison: {config.reports_dir / 'model_comparison.csv'}")
print(f"- Plot: {config.reports_dir / 'performance_comparison.png'}")
print()

print("FINAL SUMMARY:")
print(f"Best Model: {best_model_name}")
print(f"Test ROC-AUC: {test_metrics['roc_auc']:.6f}")
print(f"Models Compared: {len(comparison_results)}")
print()

print("OPTIMIZATION COMPLETED SUCCESSFULLY!")

SAVING RESULTS
RESULTS SAVED:
- Models: C:\Users\Arosha IIT\OneDrive - Robert Gordon University\Desktop\AIDS NOTES\MindScope\results\models\final_tuned_models.pkl
- Comparison: C:\Users\Arosha IIT\OneDrive - Robert Gordon University\Desktop\AIDS NOTES\MindScope\results\reports\model_comparison.csv
- Plot: C:\Users\Arosha IIT\OneDrive - Robert Gordon University\Desktop\AIDS NOTES\MindScope\results\reports\performance_comparison.png

FINAL SUMMARY:
Best Model: XGBoost
Test ROC-AUC: 0.998875
Models Compared: 4

OPTIMIZATION COMPLETED SUCCESSFULLY!
